# 📊 Würth – Geo/ORSY Analyse: Marktstruktur & ORSY-Intensität

Ziel heute:
1) **Marktstruktur** verstehen: Unternehmensgröße × Bestellaktivität – getrennt für **Nicht-ORSY** und **ORSY**  
2) **ORSY-Intensität** bewerten: Zusammenhang zwischen **ORSY-Anteil** und **Gesamtumsatz**

Hinweise:
- *ORSY-Anteil* = `sales_orsy_relevant / (rev_salesrep + rev_branch_office + rev_ebusiness + rev_internal_staff)`  
- `rev_others` wird als Korrekturposten **nicht** für Anteile genutzt.

In [ ]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA_PATH = os.path.join("..", "data", "dataset_wuerth.csv")
df = pd.read_csv(DATA_PATH)

# Flags/Labels
df["orsy_flag"] = (df["flag_new_orsyshelf"] > 0).astype(int)
df["orsy_str"]  = df["orsy_flag"].map({0:"Nicht-ORSY", 1:"ORSY"})

# 4 saubere Kanäle (ohne Others)
rev_cols4 = ["rev_salesrep", "rev_branch_office", "rev_ebusiness", "rev_internal_staff"]
df["total_rev_4ch"] = df[rev_cols4].sum(axis=1)

# ORSY-Anteil (gegen 4-Kanal-Umsatz)
df["orsy_share"] = np.where(df["total_rev_4ch"] > 0, df["sales_orsy_relevant"] / df["total_rev_4ch"], np.nan)

# Fokus auf relevante Masse: max 200 Mitarbeiter
df = df[df["emp_count"].fillna(0) <= 200].copy()

orsi_rate = df["orsy_flag"].mean()
print(f"✅ Daten: {df.shape}  |  ORSY-Rate: {orsi_rate:.1%}")

In [27]:
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import math

# ===== Basis (<= 200 MA) =====
df_c = df.copy()
df_c = df_c[(df_c["emp_count"].notna()) & (df_c["sales"].notna()) & (df_c["emp_count"] <= 200)]
df_c["orsy_flag"] = (df_c["flag_new_orsyshelf"] > 0).astype(int)
df_c["orsy_str"]  = df_c["orsy_flag"].map({0:"Nicht-ORSY", 1:"ORSY"})

# ===== Einheitliche Bins (gleiche Feldgröße) =====
emp_edges  = [-np.inf, 1, 5, 10, 20, 50, 100, 200]
emp_labels = ["0–1","2–5","6–10","11–20","21–50","51–100","101–200"]

# Umsatzklassen (anpassen bei Bedarf)
sales_edges  = [0, 500, 1_000, 2_000, 5_000, 10_000, 20_000, 50_000, 100_000, 200_000, 500_000, float(df_c["sales"].max())]
def human_label(edges):
    labs = []
    for a,b in zip(edges[:-1], edges[1:]):
        def fmt(x):
            if x >= 1_000_000: return f"{x/1_000_000:.1f} Mio"
            if x >= 1_000:     return f"{x/1_000:.0f} k"
            return f"{int(x)}"
        labs.append(f"{fmt(a)}–{fmt(b)}")
    return labs
sales_labels = human_label(sales_edges)

df_c["emp_bin"]   = pd.cut(df_c["emp_count"], bins=emp_edges,  labels=emp_labels,  include_lowest=True, right=True)
df_c["sales_bin"] = pd.cut(df_c["sales"],     bins=sales_edges, labels=sales_labels, include_lowest=True, right=True)

# ===== Aggregation: Kundenanzahl je Feld =====
agg = (df_c.groupby(["orsy_str","emp_bin","sales_bin"], observed=True)
          .size()
          .reset_index(name="customer_count"))

def full_matrix(group):
    sub = agg[agg["orsy_str"]==group]
    mat = (sub.pivot_table(index="emp_bin", columns="sales_bin", values="customer_count", fill_value=0)
             .reindex(index=emp_labels, columns=sales_labels))
    return mat.fillna(0)

mat_non  = full_matrix("Nicht-ORSY")
mat_orsi = full_matrix("ORSY")

# ===== ORSY-Rate & Skalen berechnen =====
orsy_rate = float(df_c["orsy_flag"].mean())               # ~0.09...
zmax_non  = float(np.nanmax(mat_non.values))              # z. B. 6000
expected_orsi_max = max(1.0, orsy_rate * zmax_non)        # z. B. 0.09*6000 = 540
# (Optional) auf ganze Zahl runden
expected_orsi_max = math.ceil(expected_orsi_max)

# ===== Plot: getrennte Skalen, gleiche Bins, helle große Werte =====
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(f"Nicht-ORSY", f"ORSY (Skala an {orsy_rate:.1%} angepasst)"),
    horizontal_spacing=0.18
)

t1 = px.imshow(mat_non,  origin="lower", color_continuous_scale="Viridis_r").data[0]
t2 = px.imshow(mat_orsi, origin="lower", color_continuous_scale="Viridis_r").data[0]

t1.update(coloraxis="coloraxis")
t2.update(coloraxis="coloraxis2")

fig.add_trace(t1, row=1, col=1)
fig.add_trace(t2, row=1, col=2)

fig.update_layout(
    height=600, width=1300,
    title="Kundenverteilung: Mitarbeiter (Y) × Umsatzklasse (X) – Skalen fair angepasst",
    # links: reales Maximum, rechts: erwartetes Maximum = ORSY-Rate * links-Max
    coloraxis =dict(cmin=0, cmax=zmax_non,
                    colorbar=dict(title="Kunden", len=0.85, x=0.43)),
    coloraxis2=dict(cmin=0, cmax=expected_orsi_max,
                    colorbar=dict(title="Kunden", len=0.85, x=1.02)),
    title_x=0.5, font=dict(size=13)
)

# Achsenlabels aus gemeinsamen Bins
for c in [1,2]:
    fig.update_xaxes(title_text="Umsatzklasse (€)", row=1, col=c,
                     tickmode="array", tickvals=list(range(len(sales_labels))), ticktext=sales_labels)
    fig.update_yaxes(title_text="Mitarbeiterklasse", row=1, col=c,
                     tickmode="array", tickvals=list(range(len(emp_labels))),  ticktext=emp_labels)

fig.show()

print(f"ORSY-Rate: {orsy_rate:.2%} | Max Nicht-ORSY: {int(zmax_non)} Kunden | ORSY-Skalen-Max: {expected_orsi_max} Kunden")

/var/folders/7w/3vw94kq12mbbs075sr0n34zr0000gp/T/ipykernel_42729/950002289.py:40: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior

/var/folders/7w/3vw94kq12mbbs075sr0n34zr0000gp/T/ipykernel_42729/950002289.py:40: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior



ORSY-Rate: 9.36% | Max Nicht-ORSY: 6653 Kunden | ORSY-Skalen-Max: 623 Kunden


In [37]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ===== Datenbasis & Bins =====
df_r = df.copy()
df_r = df_r[(df_r["emp_count"].notna()) & (df_r["sales"].notna()) & (df_r["emp_count"] <= 200)]
df_r["orsy_flag"] = (df_r["flag_new_orsyshelf"] > 0).astype(int)

emp_edges  = [-np.inf, 1, 5, 10, 20, 50, 100, 200]
emp_labels = ["0–1","2–5","6–10","11–20","21–50","51–100","101–200"]

sales_edges  = [0, 500, 1_000, 2_000, 5_000, 10_000, 20_000, 50_000, 100_000, 200_000, 500_000, float(df_r["sales"].max())]
def human_label(edges):
    labs=[]
    for a,b in zip(edges[:-1], edges[1:]):
        def fmt(x):
            if x >= 1_000_000: return f"{x/1_000_000:.1f} Mio €"
            if x >= 1_000:     return f"{x/1_000:.0f} Tsd €"
            return f"{int(x)} €"
        labs.append(f"{fmt(a)}–{fmt(b)}")
    return labs
sales_labels = human_label(sales_edges)

df_r["emp_bin"]   = pd.cut(df_r["emp_count"], bins=emp_edges,  labels=emp_labels,  include_lowest=True, right=True)
df_r["sales_bin"] = pd.cut(df_r["sales"],     bins=sales_edges, labels=sales_labels, include_lowest=True, right=True)

# ===== Counts je Feld =====
agg = (df_r.groupby(["orsy_flag","emp_bin","sales_bin"], observed=True)
          .size()
          .reset_index(name="n"))

def count_matrix(flag_val):
    sub = agg[agg["orsy_flag"]==flag_val]
    mat = (sub.pivot_table(index="emp_bin", columns="sales_bin", values="n", fill_value=0)
             .reindex(index=emp_labels, columns=sales_labels))
    return mat.fillna(0)

cnt_non  = count_matrix(0)
cnt_orsi = count_matrix(1)
total    = (cnt_non + cnt_orsi)

# ===== ORSY-Anteil (0..1) =====
share_vals = np.divide(
    cnt_orsi.values.astype(float), total.values,
    out=np.full_like(cnt_orsi.values, np.nan, dtype=float),
    where=total.values > 0
)
share = pd.DataFrame(share_vals, index=cnt_non.index, columns=cnt_non.columns)

# ===== Hovertext ORSY-Anteil =====
hover_text_share = []
for r in share.index:
    row = []
    for c in share.columns:
        n_non  = int(cnt_non.loc[r, c])
        n_orsi = int(cnt_orsi.loc[r, c])
        n_tot  = n_non + n_orsi
        s      = share.loc[r, c]
        if np.isnan(s):
            row.append(f"{r} × {c}<br>Keine Kunden")
        else:
            row.append(
                f"{r} × {c}"
                f"<br>ORSY: {n_orsi:,}"
                f"<br>Nicht-ORSY: {n_non:,}"
                f"<br>Gesamt: {n_tot:,}"
                f"<br>ORSY-Anteil: {s:.0%}"
            )
    hover_text_share.append(row)

# ===== Umsatzsumme je Feld (komplett in Euro) =====
agg_sales = (df_r.groupby(["emp_bin","sales_bin"], observed=True)["sales"]
               .sum().reset_index())
mat_sales = (agg_sales.pivot_table(index="emp_bin", columns="sales_bin", values="sales", fill_value=0)
               .reindex(index=emp_labels, columns=sales_labels)).fillna(0)
sales_max_eur = float(mat_sales.values.max()) if mat_sales.size else 0.0

# Tooltip rechts mit vollständigen Euro-Beträgen + ORSY-Anteil
hover_text_sales = []
for r in mat_sales.index:
    row = []
    for c in mat_sales.columns:
        sales_eur = mat_sales.loc[r, c]
        n_tot   = int(total.loc[r, c])
        s       = share.loc[r, c]
        if n_tot == 0 or np.isnan(s):
            row.append(
                f"{r} × {c}"
                f"<br>Umsatz: {sales_eur:,.0f} €"
                f"<br>Kunden: 0"
                f"<br>ORSY-Anteil: —"
            )
        else:
            row.append(
                f"{r} × {c}"
                f"<br>Umsatz: {sales_eur:,.0f} €"
                f"<br>Kunden: {n_tot:,}"
                f"<br>ORSY-Anteil: {s:.0%}"
            )
    hover_text_sales.append(row)

# ===== Subplots (mit großem Abstand & klaren Legenden) =====
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("ORSY-Anteil je Feld", "Umsatzsumme je Feld (in €)"),
    horizontal_spacing=0.25
)

# Links: ORSY-Anteil
heat_share = go.Heatmap(
    z=share.values, x=list(range(len(sales_labels))), y=list(range(len(emp_labels))),
    colorscale="Greens", zmin=0, zmax=1,
    colorbar=dict(title="ORSY-Anteil", tickformat=".0%", x=0.42, len=0.8),
    hoverinfo="text", text=hover_text_share
)
nan_mask = share.isna().astype(float).values
nan_mask[nan_mask == 0] = np.nan
heat_nan = go.Heatmap(
    z=nan_mask, x=list(range(len(sales_labels))), y=list(range(len(emp_labels))),
    colorscale=[[0,"lightgrey"], [1,"lightgrey"]],
    showscale=False, hoverinfo="skip"
)
fig.add_trace(heat_share, row=1, col=1)
fig.add_trace(heat_nan,   row=1, col=1)

# Rechts: Umsatz komplett in Euro
heat_sales = go.Heatmap(
    z=mat_sales.values, x=list(range(len(sales_labels))), y=list(range(len(emp_labels))),
    colorscale="Blues", zmin=0, zmax=sales_max_eur,
    colorbar=dict(title="Umsatz (€)", x=1.05, len=0.8),
    hoverinfo="text", text=hover_text_sales
)
fig.add_trace(heat_sales, row=1, col=2)

# Achsen
for c in [1,2]:
    fig.update_xaxes(title_text="Umsatzklasse", row=1, col=c,
                     tickmode="array", tickvals=list(range(len(sales_labels))), ticktext=sales_labels)
    fig.update_yaxes(title_text="Mitarbeiterklasse", row=1, col=c,
                     tickmode="array", tickvals=list(range(len(emp_labels))),  ticktext=emp_labels)

# Layout
fig.update_layout(
    height=650, width=1550,
    title="Vergleich: ORSY-Anteil vs. absolute Umsatzverteilung (komplette Werte in €)",
    title_x=0.5, font=dict(size=13),
    margin=dict(t=80, b=60, l=60, r=120)
)

fig.show()

print(f"Gesamter ORSY-Anteil: {df_r['orsy_flag'].mean():.2%}")
print(f"Maximales Umsatzfeld: {sales_max_eur:,.0f} €")

/var/folders/7w/3vw94kq12mbbs075sr0n34zr0000gp/T/ipykernel_42729/1955930619.py:36: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior

/var/folders/7w/3vw94kq12mbbs075sr0n34zr0000gp/T/ipykernel_42729/1955930619.py:36: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior

/var/folders/7w/3vw94kq12mbbs075sr0n34zr0000gp/T/ipykernel_42729/1955930619.py:76: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior



Gesamter ORSY-Anteil: 9.36%
Maximales Umsatzfeld: 14,132,804 €


In [47]:
import os
import numpy as np
import pandas as pd
from IPython.display import display

# === Daten laden ===
data_path = os.path.join("..", "data", "dataset_wuerth.csv")
df = pd.read_csv(data_path)

# ORSY-Flag vereinheitlichen
df["flag_new_orsyshelf"] = (df["flag_new_orsyshelf"] > 0).astype(int)

# === Distrikt-Auswertung mit Wertebereich (Min–Max) ===
def district_summary(df_sub: pd.DataFrame, value_col: str = "sales") -> dict:
    """Berechnet Umsatzkennzahlen auf Distrikt-Ebene."""
    sums = df_sub.groupby("district", observed=True)[value_col].sum().dropna()
    if len(sums) == 0:
        return {
            "Distrikte (#)": 0,
            "Ø Umsatz pro Distrikt (€, Summe)": np.nan,
            "SD Umsatz Distrikte (€, Summe)": np.nan,
            "Umsatzbereich Distrikte (€)": np.nan,
            "25%-Quantil Distrikt-Umsatz (€)": np.nan,
            "75%-Quantil Distrikt-Umsatz (€)": np.nan,
        }

    return {
        "Distrikte (#)": len(sums),
        "Ø Umsatz pro Distrikt (€, Summe)": sums.mean(),
        "SD Umsatz Distrikte (€, Summe)": sums.std(ddof=1) if len(sums) > 1 else 0.0,
        "Umsatzbereich Distrikte (€)": f"{sums.min():,.0f} – {sums.max():,.0f}",
        "25%-Quantil Distrikt-Umsatz (€)": sums.quantile(0.25),
        "75%-Quantil Distrikt-Umsatz (€)": sums.quantile(0.75),
    }

# === Haupttabelle Region × Gruppe ===
def region_table(df: pd.DataFrame) -> pd.DataFrame:
    base = (
        df.groupby(["region", "flag_new_orsyshelf"], observed=True)
          .agg(
              **{
                  "Kunden (#)": ("cust_id", "count"),
                  "Umsatz (Summe, €)": ("sales", "sum"),
                  "Ø Umsatz pro Kunde (€, mean)": ("sales", "mean"),
                  "Median Umsatz pro Kunde (€, median)": ("sales", "median"),
                  "Ø Bestellungen pro Kunde": ("orders_count", "mean"),
                  "Ø Mitarbeiter je Kunde": ("emp_count", "mean"),
              }
          )
          .reset_index()
    )

    # Distrikt-Metriken anhängen
    dist_blocks = []
    for _, row in base.iterrows():
        reg = int(row["region"])
        grp = int(row["flag_new_orsyshelf"])
        sub = df[(df["region"] == reg) & (df["flag_new_orsyshelf"] == grp)]
        dist_info = district_summary(sub, value_col="sales")
        dist_blocks.append({"region": reg, "flag_new_orsyshelf": grp, **dist_info})

    dist_df = pd.DataFrame(dist_blocks)
    out = base.merge(dist_df, on=["region", "flag_new_orsyshelf"], how="left")
    out["Gruppe"] = out["flag_new_orsyshelf"].map({0: "Nicht-ORSY", 1: "ORSY"})
    out = out.drop(columns=["flag_new_orsyshelf"])

    # Sortieren
    out = out.sort_values(["region", "Gruppe"]).reset_index(drop=True)

    # Nur numerische Spalten runden
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].round(2)

    # Zählspalten als int
    for c in ["Kunden (#)", "Distrikte (#)"]:
        if c in out.columns:
            out[c] = out[c].astype(int)

    # Spaltenreihenfolge
    cols = [
        "region", "Gruppe",
        "Kunden (#)",
        "Umsatz (Summe, €)",
        "Ø Umsatz pro Kunde (€, mean)",
        "Median Umsatz pro Kunde (€, median)",
        "Ø Bestellungen pro Kunde",
        "Ø Mitarbeiter je Kunde",
        "Distrikte (#)",
        "Ø Umsatz pro Distrikt (€, Summe)",
        "SD Umsatz Distrikte (€, Summe)",
        "Umsatzbereich Distrikte (€)",
        "25%-Quantil Distrikt-Umsatz (€)",
        "75%-Quantil Distrikt-Umsatz (€)"
    ]
    out = out[[c for c in cols if c in out.columns]]
    return out

# === Tabelle erzeugen & anzeigen ===
region_overview = region_table(df)

print("=== Region (11–18) × Gruppe – Distriktanalyse mit Umsatzbereich & Quantilen ===")
display(region_overview)

# === Export ===
out_dir = os.path.join("..", "data")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "region_overview_district_full.csv")
region_overview.to_csv(out_path, index=False)
print(f"\n✅ Export gespeichert: {out_path}")

# ===== Durchschnitts-REGION (Textzusammenfassung unter der Tabelle) =====
def _avg_region_profile(df_sub: pd.DataFrame) -> dict | None:
    """Aggregiert je Region und mittelt dann über Regionen (ungewichtet)."""
    if df_sub.empty:
        return None

    # je Region aggregieren
    per_reg = (df_sub.groupby("region", observed=True)
                      .agg(customers=("cust_id", "count"),
                           sales_sum=("sales", "sum"),
                           avg_sales_per_cust=("sales", "mean"),
                           avg_orders_per_cust=("orders_count", "mean"),
                           avg_emp_per_cust=("emp_count", "mean"))
                      .reset_index())

    # ungewichtete Mittelwerte über Regionen
    res = {
        "regions": int(per_reg.shape[0]),
        "avg_customers_per_region": float(per_reg["customers"].mean()),
        "avg_sales_sum_per_region": float(per_reg["sales_sum"].mean()),
        "avg_sales_per_customer_region_mean": float(per_reg["avg_sales_per_cust"].mean()),
        "avg_orders_per_customer_region_mean": float(per_reg["avg_orders_per_cust"].mean()),
        "avg_emp_per_customer_region_mean": float(per_reg["avg_emp_per_cust"].mean()),
        # zum Vergleich: gewichtete Gesamtschnitte (über alle Kunden)
        "weighted_sales_per_customer_all": float(df_sub["sales"].mean()),
        "weighted_orders_per_customer_all": float(df_sub["orders_count"].mean()),
        "weighted_emp_per_customer_all": float(df_sub["emp_count"].mean()),
    }
    return res

def _fmt_eur(x):  # deutsche Tausenderpunkte
    return f"{x:,.0f} €".replace(",", ".")

def _fmt_num(x):
    return f"{x:,.2f}".replace(",", ".")

def print_avg_region_summary(df: pd.DataFrame):
    groups = {
        "Nicht-ORSY": df[df["flag_new_orsyshelf"] == 0],
        "ORSY":       df[df["flag_new_orsyshelf"] == 1],
        "Gesamt":     df,  # beide zusammen
    }

    print("\n--- Durchschnittsregion (über Regionen gemittelt) ---")
    for name, sub in groups.items():
        r = _avg_region_profile(sub)
        if r is None:
            print(f"{name}: keine Daten")
            continue

        print(
            f"{name} (über {r['regions']} Regionen):\n"
            f"  • Ø Kunden je Region:           {_fmt_num(r['avg_customers_per_region'])}\n"
            f"  • Ø Umsatz je Region:            {_fmt_eur(r['avg_sales_sum_per_region'])}\n"
            f"  • Ø Umsatz je Kunde (Regionsschnitt, ungew.): {_fmt_eur(r['avg_sales_per_customer_region_mean'])}\n"
            f"  • Ø Umsatz je Kunde (gesamt, gewichtet):      {_fmt_eur(r['weighted_sales_per_customer_all'])}\n"
            f"  • Ø Bestellungen je Kunde (Regionsschnitt):   {_fmt_num(r['avg_orders_per_customer_region_mean'])}\n"
            f"  • Ø Mitarbeiter je Kunde (Regionsschnitt):    {_fmt_num(r['avg_emp_per_customer_region_mean'])}"
        )

# Aufruf direkt nach der Tabellenausgabe:
print_avg_region_summary(df)

=== Region (11–18) × Gruppe – Distriktanalyse mit Umsatzbereich & Quantilen ===


,region,Gruppe,Kunden (#),"Umsatz (Summe, €)","Ø Umsatz pro Kunde (€, mean)","Median Umsatz pro Kunde (€, median)",Ø Bestellungen pro Kunde,Ø Mitarbeiter je Kunde,Distrikte (#),"Ø Umsatz pro Distrikt (€, Summe)","SD Umsatz Distrikte (€, Summe)",Umsatzbereich Distrikte (€),25%-Quantil Distrikt-Umsatz (€),75%-Quantil Distrikt-Umsatz (€)
0,11,Nicht-ORSY,3387,9560893.10,2822.82,815.53,15.80,4.16,24,398370.55,227536.36,"20 – 879,873",300667.26,506143.17
1,11,ORSY,330,7322648.18,22189.84,14308.82,94.57,7.10,21,348697.53,165276.70,"135,108 – 715,236",206815.23,476139.55
2,12,Nicht-ORSY,3181,8633644.84,2714.13,748.36,15.37,3.98,20,431682.24,203610.55,"30 – 789,241",349448.55,529804.64
3,12,ORSY,298,6258874.21,21002.93,12715.35,87.05,7.21,18,347715.23,184561.77,"88,825 – 729,389",221341.71,434521.62
4,13,Nicht-ORSY,4127,11905246.61,2884.72,764.44,15.97,4.52,23,517619.42,259495.13,"3,665 – 1,049,553",324304.76,696604.86
5,13,ORSY,409,8407389.06,20555.96,12137.45,88.44,7.59,21,400351.86,167858.08,"101,667 – 856,942",295055.33,474515.08
6,14,Nicht-ORSY,2760,6819815.48,2470.95,776.16,14.88,3.93,18,378878.64,199038.61,"163 – 674,442",330844.33,516963.27
7,14,ORSY,293,5640457.80,19250.71,11510.18,83.39,6.78,15,376030.52,153993.58,"117,292 – 680,209",287887.80,465648.22
8,15,Nicht-ORSY,3856,10133678.52,2628.03,705.50,14.69,4.16,23,440594.72,255473.11,"1,277 – 950,110",302287.26,580811.88
9,15,ORSY,371,8195014.16,22088.99,13421.39,84.33,6.95,20,409750.71,239232.81,"136,124 – 900,716",211059.50,561024.69



✅ Export gespeichert: ../data/region_overview_district_full.csv

--- Durchschnittsregion (über Regionen gemittelt) ---
Nicht-ORSY (über 8 Regionen):
  • Ø Kunden je Region:           3.341.75
  • Ø Umsatz je Region:            9.189.338 €
  • Ø Umsatz je Kunde (Regionsschnitt, ungew.): 2.736 €
  • Ø Umsatz je Kunde (gesamt, gewichtet):      2.750 €
  • Ø Bestellungen je Kunde (Regionsschnitt):   15.06
  • Ø Mitarbeiter je Kunde (Regionsschnitt):    4.05
ORSY (über 8 Regionen):
  • Ø Kunden je Region:           344.88
  • Ø Umsatz je Region:            7.153.930 €
  • Ø Umsatz je Kunde (Regionsschnitt, ungew.): 20.737 €
  • Ø Umsatz je Kunde (gesamt, gewichtet):      20.744 €
  • Ø Bestellungen je Kunde (Regionsschnitt):   84.67
  • Ø Mitarbeiter je Kunde (Regionsschnitt):    7.11
Gesamt (über 8 Regionen):
  • Ø Kunden je Region:           3.686.62
  • Ø Umsatz je Region:            16.343.268 €
  • Ø Umsatz je Kunde (Regionsschnitt, ungew.): 4.429 €
  • Ø Umsatz je Kunde (gesamt, gewi